# Workshop 05 — Deploy the Fine-tuned Qwen3-4B via Bedrock Custom Model Import

> **⚠️ Run this in your OWN AWS account, not the shared workshop account.** The workshop account does not permit creating Bedrock custom models (`bedrock:CreateModelImportJob` is blocked by an account guardrail/SCP), so this notebook is a **take-home / post-workshop exercise**: copy your fine-tuned weights to an account where you have Bedrock Custom Model Import permissions and run it there.

This replaces the SageMaker real-time endpoint from notebook 01 with **Amazon Bedrock Custom Model Import (CMI)** — serverless, pay-per-use, invoked through the standard Bedrock runtime.

## What you'll do

1. **Point at the merged weights** the SFT job already wrote (`.../checkpoints/hf_merged/`) — no LoRA merge needed; CMI imports the HF-format weights directly.
2. **Reuse the SageMaker execution role** as the import role (must be trusted by `bedrock.amazonaws.com`).
3. **`create_model_import_job`** and poll until `Completed`.
4. **Invoke** the imported model with `bedrock-runtime` `invoke_model`.

## Why Bedrock CMI instead of a SageMaker endpoint

| | SageMaker real-time endpoint (nb 01) | Bedrock Custom Model Import (this nb) |
|---|---|---|
| Billing | per-hour while the endpoint runs | per-token + per-copy on-demand; scales to zero |
| Ops | you manage the endpoint lifecycle | fully managed |
| Invoke API | `sagemaker-runtime.invoke_endpoint` | `bedrock-runtime.invoke_model` |

## ⚠️ Hard requirements & caveats (verified against AWS docs)

- **Permissions:** the account/role needs `bedrock:CreateModelImportJob` + friends and `iam:PassRole` to `bedrock.amazonaws.com`. **This is exactly what the workshop account blocks** — hence run elsewhere.
- **Architecture:** Qwen3 is supported, but **only `Qwen3ForCausalLM`** (and `Qwen3MoeForCausalLM`). Our SFT-LoRA output is `Qwen3ForCausalLM` ✓.
- **No Converse API for Qwen3** — you must use `InvokeModel` (we do).
- **Transformers 4.51.3** — CMI expects weights produced with transformers 4.51.3.
- **Merged weights only** — CMI imports full weights, not a bare LoRA adapter. The notebook-01 SFT job already wrote merged weights (`hf_merged/`), so we skip the merge entirely.
- **Size limits:** < 200 GB for text models, context < 128K. Qwen3-4B is ~8 GB — fine.
- **Regions:** CMI is in `us-east-1`, `us-east-2`, `us-west-2`, `eu-central-1`.
- Tokenizer must be supported — `Qwen2Tokenizer`/`Qwen2TokenizerFast` ✓.

## 0. Install / upgrade

In [ ]:
%pip install -q --upgrade boto3 sagemaker datasets
# No merge needed — the SFT job already wrote fully-merged HF weights (hf_merged/).
# This notebook only imports those into Bedrock and invokes them.
# Restart the kernel after this cell.

## 1. Configuration

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)
BUCKET = sagemaker_session.default_bucket()

# The SFT training job (notebook 01) already wrote fully-MERGED HF weights here.
# No LoRA merge step needed — point Bedrock straight at this prefix.
SFT_JOB_NAME = "huggingface-reasoning-qwen3-4b-sft-20260624052156"  # from notebook 01
MERGED_S3_URI = f"s3://{BUCKET}/qwen3-4b-phish-sft/{SFT_JOB_NAME}/output/model/checkpoints/hf_merged/"

IMPORTED_MODEL_NAME = "qwen3-4b-phish-sft"
IMPORT_JOB_NAME = "qwen3-4b-phish-import-01"
ASSERT_REGION_OK = REGION in {"us-east-1", "us-east-2", "us-west-2", "eu-central-1"}
print(f"region={REGION}  cmi_supported={ASSERT_REGION_OK}  bucket={BUCKET}")
assert ASSERT_REGION_OK, "Bedrock Custom Model Import is not available in this region."

# Identity / execution role (used as the Bedrock import roleArn below).
_ident = boto3.client("sts").get_caller_identity()
account = _ident["Account"]
caller_arn = _ident["Arn"]
role_arn = caller_arn
if ":assumed-role/" in caller_arn:
    role_arn = f"arn:aws:iam::{account}:role/" + caller_arn.split("/")[1]
print("execution role:", role_arn)

## 2. Point at the already-merged weights

Notebook 01's SFT job wrote **fully-merged HF weights** to `.../output/model/checkpoints/hf_merged/` — `config.json`, sharded `.safetensors`, and the tokenizer. **No LoRA merge step is needed**; Bedrock imports this prefix directly.

This cell just verifies the prefix has Bedrock-ready files (a `config.json` whose architecture is `Qwen3ForCausalLM`, `.safetensors`, and tokenizer files).

In [ ]:
import json
from urllib.parse import urlparse
s3 = boto3.client('s3')
u = urlparse(MERGED_S3_URI.rstrip('/'))
mb, mp = u.netloc, u.path.lstrip('/')
files = []
for page in s3.get_paginator('list_objects_v2').paginate(Bucket=mb, Prefix=mp + '/'):
    files += [o['Key'].split('/')[-1] for o in page.get('Contents', [])]
print('files:', sorted(f for f in files if f))

assert any(f.endswith('.safetensors') for f in files), 'no .safetensors at MERGED_S3_URI'
assert 'config.json' in files, 'no config.json at MERGED_S3_URI'
assert any(f.startswith('tokenizer') for f in files), 'no tokenizer files at MERGED_S3_URI'

cfg = json.loads(s3.get_object(Bucket=mb, Key=mp + '/config.json')['Body'].read())
arch = (cfg.get('architectures') or [''])[0]
print('architecture:', arch, '| max_pos_emb:', cfg.get('max_position_embeddings'))
assert arch in ('Qwen3ForCausalLM', 'Qwen3MoeForCausalLM'), f'Bedrock CMI unsupported arch: {arch}'
print('\nReady to import:', MERGED_S3_URI)

## 3. Use the SageMaker execution role for the import

Bedrock assumes a role to read your S3 weights. Rather than create a dedicated import service role, we **reuse the SageMaker execution role** — it's already trusted by `bedrock.amazonaws.com` (the trust policy was set up to allow it) and already has S3 access to the bucket holding the merged weights.

The cell below confirms the trust relationship includes `bedrock.amazonaws.com` before using it, so you get a clear message if it doesn't (in which case fall back to a dedicated service role: trust `bedrock.amazonaws.com` + `s3:GetObject`/`s3:ListBucket` on the bucket).

In [ ]:
import json
iam = boto3.client("iam")

# Reuse the execution role resolved in section 2 (role_arn).
IMPORT_ROLE_ARN = role_arn
exec_role_name = role_arn.split("/")[-1]

# Verify the trust policy allows bedrock.amazonaws.com to assume this role.
try:
    rd = iam.get_role(RoleName=exec_role_name)["Role"]
    trust = rd["AssumeRolePolicyDocument"]
    principals = []
    for st in trust.get("Statement", []):
        svc = st.get("Principal", {}).get("Service")
        principals += [svc] if isinstance(svc, str) else (svc or [])
    if any("bedrock.amazonaws.com" in pr for pr in principals):
        print(f"OK — {exec_role_name} is trusted by bedrock.amazonaws.com")
    else:
        print(f"\u26a0\ufe0f  {exec_role_name} trust principals = {principals}")
        print("   bedrock.amazonaws.com is NOT in the trust policy. The import job will fail")
        print("   with AccessDenied/cannot-assume. Add bedrock.amazonaws.com to this role's")
        print("   trust relationship, or create a dedicated import role (see markdown above).")
except iam.exceptions.ClientError as e:
    print(f"Could not read role trust policy ({e}). Proceeding; the import job will tell us.")

print("import roleArn:", IMPORT_ROLE_ARN)

## 4. Create the import job + poll

In [ ]:
import time
bedrock = boto3.client("bedrock", region_name=REGION)

create = bedrock.create_model_import_job(
    jobName=IMPORT_JOB_NAME,
    importedModelName=IMPORTED_MODEL_NAME,
    roleArn=IMPORT_ROLE_ARN,
    modelDataSource={"s3DataSource": {"s3Uri": MERGED_S3_URI}},
)
job_arn = create["jobArn"]
print("import job:", job_arn)

# Poll (typically 10-25 min for a 4B model)
while True:
    j = bedrock.get_model_import_job(jobIdentifier=job_arn)
    status = j["status"]
    print(f"  status={status}")
    if status in ("Completed", "Failed"):
        if status == "Failed":
            print("failureMessage:", j.get("failureMessage"))
        break
    time.sleep(30)

assert status == "Completed", "import failed — see failureMessage above"
imported_model_arn = j["importedModelArn"]
print("\nimported model ARN:", imported_model_arn)

## 5. Invoke the imported model

Use `bedrock-runtime.invoke_model` with the imported model ARN as `modelId`. **Converse is not supported for Qwen3**, so we use the raw `InvokeModel` text-completion schema.

First invocation cold-starts a model copy (can take ~1 min); subsequent calls are fast.

In [ ]:
import json
brt = boto3.client("bedrock-runtime", region_name=REGION)

def classify(prompt: str, max_tokens: int = 24) -> str:
    body = {"prompt": prompt, "max_tokens": max_tokens, "temperature": 0.0, "top_p": 1.0}
    resp = brt.invoke_model(modelId=imported_model_arn, body=json.dumps(body),
                            accept="application/json", contentType="application/json")
    out = json.loads(resp["body"].read())
    # CMI text models return OpenAI-ish completion JSON; handle common shapes
    if isinstance(out, dict):
        if "generation" in out:
            return out["generation"]
        if "choices" in out and out["choices"]:
            ch = out["choices"][0]
            return ch.get("text") or ch.get("message", {}).get("content", "")
        if "outputs" in out and out["outputs"]:
            return out["outputs"][0].get("text", "")
    return json.dumps(out)

# Retry once for cold start
import botocore
demo_prompt = (
    "You are a phishing-detection classifier. Given a webpage's URL and a compact summary "
    "of its content, output exactly one word: 'phish' or 'benign'.\n\n---\n"
    "URL: https://paypa1-secure-login.example.com/verify\nTitle: PayPal - Confirm your account\n"
    "Forms: form action=https://evil.example.com/steal fields=[email, password]\n---\n\nLabel:"
)
for attempt in range(3):
    try:
        print(classify(demo_prompt))
        break
    except botocore.exceptions.ClientError as e:
        if "ModelNotReady" in str(e) and attempt < 2:
            print("cold start, retrying in 30s..."); time.sleep(30)
        else:
            raise

## 6. (Optional) Score the held-out set through Bedrock

Run the same validation slice notebooks 03/04 use, against the Bedrock-hosted model, parsing the reasoning output the same way (`<think>…</think>` then the label).

In [ ]:
import re, ast
from datasets import load_dataset

_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_THINK_OPEN = re.compile(r"<think>.*", re.DOTALL | re.IGNORECASE)
_LABEL = re.compile(r"(phish|benign)", re.IGNORECASE)

def extract_label(text):
    if not text:
        return ""
    try:
        v = ast.literal_eval(text)
        if isinstance(v, list) and v:
            text = str(v[0])
    except Exception:
        pass
    text = _THINK.sub(" ", text); text = _THINK_OPEN.sub(" ", text)
    m = _LABEL.search(text)
    return m.group(1).lower() if m else ""

val = load_dataset("gt2026workshop/phreshphish-2k", name="text", split="validation").select(range(20))
correct = 0
for r in val:
    pred = extract_label(classify(r["prompt"]))
    ok = pred == r["completion"].strip().lower()
    correct += ok
    print(f"{'✓' if ok else '✗'}  truth={r['completion']:<7} pred={pred or '??'}")
print(f"\naccuracy on 20 samples via Bedrock: {correct}/20 = {correct/20:.0%}")

## 7. Clean up

Imported models incur storage + per-copy charges. Delete when done:

```python
# bedrock.delete_imported_model(modelIdentifier=IMPORTED_MODEL_NAME)
```

No IAM role to clean up — we reused the existing SageMaker execution role.